# Edge-IIoTset Intrusion Detection - GWO 

In [1]:
!pip install scikit-learn
!pip install mealpy
!pip install Boruta
!pip install lightgbm
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display
from typing import List, Tuple, Dict, Optional
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, make_scorer, confusion_matrix, ConfusionMatrixDisplay, classification_report
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import LogisticRegression
from sklearn.impute import SimpleImputer
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.neural_network import MLPClassifier
from sklearn.feature_selection import VarianceThreshold
_HAS_MEALPY = False
_HAS_BORUTA = False
try:
    from boruta import BorutaPy
    _HAS_BORUTA = True
    print('BorutaPy available')
except Exception:
    _HAS_BORUTA = False
    print('BorutaPy not available yet. Run the install cell, then re-run this cell.')
try:
    from mealpy.swarm_based import GWO, BA
    _HAS_MEALPY = True
    print("Mealpy available: GWO/BA")
except Exception:
    _HAS_MEALPY = False
    print("Mealpy not available. Using fallback feature selection.")

_HAS_FLOATVAR = False
try:
    from mealpy import FloatVar
    _HAS_FLOATVAR = True
except Exception:
    _HAS_FLOATVAR = False

plt.style.use('seaborn-v0_8-darkgrid')

BorutaPy available


Mealpy available: GWO/BA


## Feature Selection Functions (GWO + Original BorutaPy + Optional BA Helpers + Without FS)

In [2]:
def _evaluate_subset(
    X: np.ndarray, y: np.ndarray, idx: List[int],
    clf=None, cv: int = 3, scoring: str = 'accuracy',
    lam: float = 0.01,
    alpha_fp: float = 0.0,   # set >0 later if you want explicit false-positive penalty
    benign_id: Optional[int] = None
) -> float:
    if len(idx) == 0:
        return 0.0

    # CHANGED: default classifier now class-weighted for robustness
    if clf is None:
        clf = DecisionTreeClassifier(random_state=42, class_weight='balanced')

    pipe = Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler(with_mean=True)),
        ('clf', clf)
    ])

    X_sub = X[:, idx]
    cvk = StratifiedKFold(n_splits=cv, shuffle=True, random_state=42)

    scorer = 'accuracy' if scoring == 'accuracy' else make_scorer(f1_score, average='macro')

    # Base CV score
    scores = cross_val_score(pipe, X_sub, y, scoring=scorer, cv=cvk, n_jobs=None)
    base = float(np.mean(scores))

    # Compactness penalty
    size_penalty = lam * (len(idx) / X.shape[1])

    # Optional false-positive penalty (OFF by default)
    fp_penalty = 0.0
    if alpha_fp > 0.0 and benign_id is not None:
        fprs = []
        for tr, te in cvk.split(X_sub, y):
            pipe.fit(X_sub[tr], y[tr])
            p = pipe.predict(X_sub[te])
            y_te = y[te]
            benign = (y_te == benign_id)
            fp = np.sum((p != benign_id) & benign)
            tn = np.sum((p == benign_id) & benign)
            fprs.append(fp / (fp + tn + 1e-12))
        fp_penalty = alpha_fp * float(np.mean(fprs))

    return base - size_penalty - fp_penalty

# =========================================
# Convergence History Storage (Global)
# =========================================
convergence_history = {}

def _optimize_continuous(
    OptimizerClass, X: np.ndarray, y: np.ndarray, k: int,
    clf=None, epochs: int = 30, pop_size: int = 30,
    cv: int = 3, scoring: str = 'accuracy', seed: int = 42
) -> Tuple[List[int], float]:

    dim = X.shape[1]
    lb_vec = [0.0] * dim
    ub_vec = [1.0] * dim

    lambda_max = 0.05
    lambda_min = 0.005

    total_calls = epochs * pop_size
    call_counter = {'n': 0}

    history = []

    def fitness(w):
        call_counter['n'] += 1
        progress = min(call_counter['n'] / total_calls, 1.0)
        t = progress * epochs

        a_t = 2 - 2 * (t / epochs)
        lam_dynamic = lambda_min + (lambda_max - lambda_min) * (a_t / 2)

        w = np.asarray(w)
        idx = list(range(dim)) if k >= dim else list(np.argsort(-w)[:k])

        score = _evaluate_subset(
            X, y, idx,
            clf=clf,
            cv=cv,
            scoring=scoring,
            lam=lam_dynamic
        )

        history.append(score)
        return score

    random.seed(int(seed))
    np.random.seed(int(seed))

    problem_mealpy = {
        'obj_func': fitness,
        'minmax': 'max',
        'lb': lb_vec,
        'ub': ub_vec,
        'seed': int(seed)
    }

    if _HAS_FLOATVAR:
        problem_mealpy['bounds'] = FloatVar(lb=lb_vec, ub=ub_vec)

    if isinstance(OptimizerClass, type):
        opt = OptimizerClass(epoch=epochs, pop_size=pop_size)
        res = opt.solve(problem_mealpy)
        algo_name = OptimizerClass.__name__
    else:
        if hasattr(OptimizerClass, 'OriginalGWO'):
            opt = OptimizerClass.OriginalGWO(epoch=epochs, pop_size=pop_size)
            algo_name = "GWO"
        elif hasattr(OptimizerClass, 'OriginalBA'):
            opt = OptimizerClass.OriginalBA(epoch=epochs, pop_size=pop_size)
            algo_name = "BA"
        else:
            raise TypeError('Unsupported optimizer type')
        res = opt.solve(problem_mealpy)

    best_pos = res[0] if isinstance(res, tuple) else res.solution

    # Save convergence history
    convergence_history[f"{algo_name}_k{k}"] = history

    w = np.asarray(best_pos)
    idx = list(range(dim)) if k >= dim else list(np.argsort(-w)[:k])

    final_lambda = lambda_min
    score = _evaluate_subset(
        X, y, idx,
        clf=clf,
        cv=cv,
        scoring=scoring,
        lam=final_lambda
    )

    return idx, score


def select_features_gwo(X, y, k, clf=None, epochs=30, pop_size=30, cv=3, scoring='accuracy', seed: int = 42):
    if _HAS_MEALPY:
        return _optimize_continuous(GWO, X, y, k, clf, epochs, pop_size, cv, scoring, seed=seed)

    from sklearn.feature_selection import mutual_info_classif
    mi = mutual_info_classif(X, y, random_state=42)
    idx = list(np.argsort(-mi)[:k])

    # CHANGED: apply penalty in fallback too
    return idx, _evaluate_subset(X, y, idx, clf=clf, cv=cv, scoring=scoring, lam=0.01)


def select_features_ba(X, y, k, clf=None, epochs=30, pop_size=30, cv=3, scoring='accuracy'):
    if _HAS_MEALPY:
        return _optimize_continuous(BA, X, y, k, clf, epochs, pop_size, cv, scoring)

    from sklearn.feature_selection import mutual_info_classif
    mi = mutual_info_classif(X, y, random_state=42)
    idx = list(np.argsort(-mi)[:k])

    # CHANGED: apply penalty in fallback too
    return idx, _evaluate_subset(X, y, idx, clf=clf, cv=cv, scoring=scoring, lam=0.01)


def select_features_boruta(X, y, k=None, clf=None, cv=3, scoring='accuracy', seed: int = 42):
    if not _HAS_BORUTA:
        raise ImportError("BorutaPy is not available. Install Boruta, then rerun the imports cell.")

    X_imp = SimpleImputer(strategy='median').fit_transform(X)
    y_arr = np.asarray(y)
    dim = int(X_imp.shape[1])

    rf = RandomForestClassifier(
   n_estimators=800,
    max_depth=30,
    min_samples_split=3,
    min_samples_leaf=1,
    max_features='sqrt',
    class_weight='balanced',
    max_samples=0.8,
    n_jobs=-1,
    random_state=42
    )

    boruta = BorutaPy(
        estimator=rf,
        n_estimators='auto',
        perc=100,
        alpha=0.05,
        two_step=True,
        max_iter=15,
        random_state=int(seed),
        verbose=0
    )
    boruta.fit(X_imp, y_arr)

    idx = list(np.where(np.asarray(boruta.support_, dtype=bool))[0])
    if len(idx) == 0:
        ranking = np.asarray(getattr(boruta, 'ranking_', np.ones(dim)), dtype=float)
        idx = [int(np.argmin(ranking))]

    return idx, _evaluate_subset(X, y_arr, idx, clf=clf, cv=cv, scoring=scoring, lam=0.0)


def select_features_hybrid_boruta(X, y, k, clf=None, cv=3, scoring='accuracy', seed: int = 42):
    X_imp = SimpleImputer(strategy='median').fit_transform(X)
    y_arr = np.asarray(y)
    dim = int(X_imp.shape[1])
    kk = int(min(max(int(k), 1), dim))

    rng = np.random.default_rng(int(seed))
    n_iter = int(max(10, int(cv) * 3))
    wins = np.zeros(dim, dtype=float)
    imp_sum = np.zeros(dim, dtype=float)

    for _ in range(n_iter):
        X_shadow = X_imp.copy()
        for j in range(dim):
            rng.shuffle(X_shadow[:, j])
        X_aug = np.hstack([X_imp, X_shadow])

        rf = RandomForestClassifier(
       n_estimators=800,
    max_depth=30,
    min_samples_split=3,
    min_samples_leaf=1,
    max_features='sqrt',
    class_weight='balanced',
    max_samples=0.8,
    n_jobs=-1,
    random_state=42
        )
        rf.fit(X_aug, y_arr)
        imp = np.asarray(getattr(rf, 'feature_importances_', None), dtype=float)
        if imp is None or imp.size != (2 * dim):
            raise RuntimeError('rf importances not available')

        real_imp = imp[:dim]
        shadow_imp = imp[dim:]
        thr = float(np.max(shadow_imp))
        wins += (real_imp > thr).astype(float)
        imp_sum += real_imp

    score = wins + 1e-3 * (imp_sum / float(n_iter))
    idx = list(np.argsort(-score)[:kk])
    return idx, _evaluate_subset(X, y_arr, idx, clf=clf, cv=cv, scoring=scoring, lam=0.01)


def select_features_without_fs(X, y, clf=None, cv=3, scoring='accuracy'):
    idx = list(range(X.shape[1]))
    return idx, _evaluate_subset(X, y, idx, clf=clf, cv=cv, scoring=scoring, lam=0.0)


def select_features_lightgbm(X, y, k, clf=None, cv=3, scoring='accuracy', seed: int = 42):
    try:
        from lightgbm import LGBMClassifier
    except ImportError:
        raise ImportError('lightgbm is not available.')
    X_imp = SimpleImputer(strategy='median').fit_transform(X)
    y_arr = np.asarray(y)
    kk = int(min(max(int(k), 1), X_imp.shape[1]))
    n_classes = len(np.unique(y_arr))
    objective = 'binary' if n_classes <= 2 else 'multiclass'
    model = LGBMClassifier(
        n_estimators=800,
        learning_rate=0.05,
        num_leaves=31,
        subsample=0.8,
        colsample_bytree=0.8,
        objective=objective,
        class_weight='balanced',
        random_state=int(seed),
        verbosity=-1
    )
    model.fit(X_imp, y_arr)
    imp = np.asarray(getattr(model, 'feature_importances_', np.zeros(X_imp.shape[1])), dtype=float)
    if imp.size != X_imp.shape[1] or np.all(imp == 0):
        raise RuntimeError('LightGBM importances not available')
    idx = list(np.argsort(-imp)[:kk])
    return idx, _evaluate_subset(X, y_arr, idx, clf=clf, cv=cv, scoring=scoring, lam=0.01)


def run_all_algorithms(X, y, ks: List[int], clf=None, epochs=30, pop_size=30, cv=3, feature_names: Optional[List[str]] = None, scoring='accuracy') -> Dict[str, Dict[int, Dict]]:
    results = {'GWO': {}, 'BA': {}, 'LightGBM': {}, 'Boruta': {}}
    for k in ks:
        idx_gwo, s_gwo = select_features_gwo(X, y, k, clf, epochs, pop_size, cv, scoring)
        idx_ba, s_ba = select_features_ba(X, y, k, clf, epochs, pop_size, cv, scoring)
        idx_lgbm, s_lgbm = select_features_lightgbm(X, y, k, clf, cv=cv, scoring=scoring)
        idx_boruta, s_boruta = select_features_boruta(X, y, None, clf, cv=cv, scoring=scoring, seed=42)

        for algo, idx, score in [('GWO', idx_gwo, s_gwo), ('BA', idx_ba, s_ba), ('LightGBM', idx_lgbm, s_lgbm), ('Boruta', idx_boruta, s_boruta)]:
            feats = [feature_names[i] for i in idx] if feature_names else None
            results[algo][k] = {'indices': idx, 'score': score, 'features': feats}

    return results

## Load Data + Cleaning + Dimensionality Reduction

In [3]:
print('Loading and preprocessing dataset...')
all_data = r"/root/iiot-run/data/cleaned_Edge-IIoTset_aligned.csv"

print('Loading data...')
combined_data = pd.read_csv(all_data, on_bad_lines='skip')
#combined_data = pd.read_csv(all_data, on_bad_lines='skip').sample(n=3000, random_state=42)

print(f"Dataset loaded successfully!")
print(f"   - Total samples: {len(combined_data)}")
print(f"   - Features: {combined_data.shape[1] - 1}")

# -------- Detect label column (label2 preferred, else label1)
label_col_src = None
for col in combined_data.columns:
    cl = col.lower()
    if 'label2' in cl:
        label_col_src = col
        break
    if 'label1' in cl and label_col_src is None:
        label_col_src = col

if label_col_src:
    label_percent = (combined_data[label_col_src].value_counts(normalize=True) * 100).round(2)
    print('\nLabel distribution (%):')
    print(label_percent)

# -------- Drop non-numeric string columns but keep label columns
string_columns = combined_data.select_dtypes(include=['object']).columns.tolist()
for keep in ['label2', 'label1', label_col_src]:
    if keep in string_columns:
        string_columns.remove(keep)

print(f"   - Dropping {len(string_columns)} string columns for ML processing")
combined_data_numeric = combined_data.drop(columns=string_columns)

target_col = 'label2' if 'label2' in combined_data_numeric.columns else ('label1' if 'label1' in combined_data_numeric.columns else None)
if target_col is None:
    raise ValueError('No label2/label1 column found in dataset')

X_raw = combined_data_numeric.drop(target_col, axis=1)
y_raw = combined_data_numeric[target_col]

# -------- Multiclass encoding
y_clean = y_raw.astype(str).str.strip()
classes = sorted(y_clean.unique())
class_to_id = {c: i for i, c in enumerate(classes)}
y = y_clean.map(class_to_id).astype(int)
print(f'Encoding used (multiclass): {class_to_id}')

# -------- Fill numeric NaNs with column mean
numeric_columns = X_raw.select_dtypes(include=[np.number]).columns
X_raw[numeric_columns] = X_raw[numeric_columns].fillna(X_raw[numeric_columns].mean())

# -------- Build df for cleaning
df = X_raw.copy()
df['label'] = y
label_col = 'label'
print(f'Dataset: {df.shape[0]} rows, {df.shape[1]} cols')
print('Target column:', label_col)

# -------- Robust time/order split (use Timestamp if present; else use row order)
use_time_split = ('Timestamp' in df.columns)

if use_time_split:
    df['__ts'] = pd.to_datetime(df['Timestamp'], errors='coerce')
    df = df.sort_values('__ts')
else:
    df = df.reset_index(drop=True)

# -------- Drop identifiers (keep Timestamp only for sorting; do not use it as a feature)
drop_cols = ['Protocol', 'Flow_ID', 'Src_IP', 'Dst_IP', 'Src_Port', 'Dst_Port', 'Timestamp']
df = df.drop(columns=[c for c in drop_cols if c in df.columns], errors='ignore')

# -------- Feature matrix and target
X_df = df.drop(columns=[label_col]).copy()
y = df[label_col].values

# -------- Convert to numeric + clean inf/nan
for col in X_df.columns:
    X_df[col] = pd.to_numeric(X_df[col], errors='coerce')

X_df.replace([np.inf, -np.inf], np.nan, inplace=True)

# -------- Fix special columns if present
for special in ['Flow_Duration', 'Fwd_Header_Length']:
    if special in X_df.columns:
        bad = (X_df[special] <= 0) | pd.isna(X_df[special])
        med = X_df[special][X_df[special] > 0].median()
        X_df.loc[bad, special] = med if not np.isnan(med) else 0

# -------- Drop columns with too many NaNs
missing_ratio = X_df.isna().mean()
X_df = X_df.loc[:, missing_ratio <= 0.4]

# -------- Fill remaining NaNs
X_df = X_df.apply(lambda c: c.fillna(c.median()))
X_df = X_df.fillna(0)

# -------- Remove zero-variance features
vt = VarianceThreshold(threshold=0.0)
vt.fit(X_df.values)
X_df = X_df.iloc[:, vt.get_support(indices=True)]

# -------- Final valid rows
valid = ~pd.isna(y)
X_df, y = X_df.loc[valid], y[valid]

feature_names = list(X_df.columns)

# -------- Split: train first 80%, test last 20% (robust generalisation test)
# split_point = int(len(X_df) * 0.8)
# X_train_df = X_df.iloc[:split_point].copy()
# X_test_df  = X_df.iloc[split_point:].copy()
# y_train = y[:split_point]
# y_test  = y[split_point:]

#new
from sklearn.model_selection import train_test_split

X_train_df, X_test_df, y_train, y_test = train_test_split(
    X_df,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)




X = X_train_df.values.astype(float)
y = y_train
X_test = X_test_df.values.astype(float)

print(f'After cleaning:')
print(f'   - Train: {X.shape[0]} rows, {X.shape[1]} columns')
print(f'   - Test:  {X_test.shape[0]} rows, {X_test.shape[1]} columns')
print(f'   - NaNs in X_train: {np.isnan(X).sum()}')
print(f'   - NaNs in X_test:  {np.isnan(X_test).sum()}')

print(np.bincount(y_train))
print(np.bincount(y_test))


Loading and preprocessing dataset...
Loading data...


Dataset loaded successfully!
   - Total samples: 104312
   - Features: 37

Label distribution (%):
label2
Ransomware               10.47
DDoS_HTTP                10.12
SQL_injection             9.88
Uploading                 9.84
DDoS_TCP                  9.82
Backdoor                  9.77
Vulnerability_scanner     9.66
Port_Scanning             9.65
Password                  9.58
XSS                       9.06
MITM                      1.16
Fingerprinting            0.96
Name: proportion, dtype: float64
   - Dropping 1 string columns for ML processing
Encoding used (multiclass): {'Backdoor': 0, 'DDoS_HTTP': 1, 'DDoS_TCP': 2, 'Fingerprinting': 3, 'MITM': 4, 'Password': 5, 'Port_Scanning': 6, 'Ransomware': 7, 'SQL_injection': 8, 'Uploading': 9, 'Vulnerability_scanner': 10, 'XSS': 11}
Dataset: 104312 rows, 37 cols
Target column: label


/tmp/ipykernel_121829/907619554.py:28: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  string_columns = combined_data.select_dtypes(include=['object']).columns.tolist()


After cleaning:
   - Train: 83449 rows, 24 columns
   - Test:  20863 rows, 24 columns
   - NaNs in X_train: 0
   - NaNs in X_test:  0
[8156 8449 8197  801  971 7991 8057 8740 8249 8215 8061 7562]
[2039 2112 2050  200  243 1998 2014 2185 2062 2054 2015 1891]


## Classification, Feature Count, Time, and Stability Evaluation

In [4]:
import random
import time
import tracemalloc
from itertools import combinations
from sklearn.base import clone
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, f1_score, matthews_corrcoef, precision_score, recall_score
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier

pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)
pd.set_option('display.width', None)

N_RUNS = 10
RUN_SEEDS = list(range(N_RUNS))
K_SUP = 10
N_FEATURES_TOTAL = int(X.shape[1])

FS_MAX_SAMPLES = 50000
FS_EPOCHS = 30
FS_POP_SIZE = 10
FS_CV_FOLDS = 3

FS_FITNESS_MODEL = 'DecisionTree'
FS_FITNESS_SCORING = 'f1_macro'

EVAL_CV_FOLDS = 3

def _subsample_for_fs_local(X_in: np.ndarray, y_in: np.ndarray, max_samples: int, seed: int):
    if (max_samples is None) or (X_in.shape[0] <= int(max_samples)):
        return X_in, y_in
    rng = np.random.default_rng(int(seed))
    idx = rng.choice(X_in.shape[0], size=int(max_samples), replace=False)
    return X_in[idx], y_in[idx]

def evaluate_cv_eff(clf, X_sub: np.ndarray, y_sub: np.ndarray, cv_folds: int, seed: int):
    cvk = StratifiedKFold(n_splits=int(cv_folds), shuffle=True, random_state=int(seed))

    accs, precs, recs, f1s, mccs = [], [], [], [], []
    train_time_s = 0.0
    infer_time_ms = 0.0

    tracemalloc.start()
    for tr, te in cvk.split(X_sub, y_sub):
        model = Pipeline([
            ('imputer', SimpleImputer(strategy='median')),
            ('scaler', StandardScaler(with_mean=True)),
            ('clf', clone(clf))
        ])

        t0 = time.perf_counter()
        model.fit(X_sub[tr], y_sub[tr])
        train_time_s += (time.perf_counter() - t0)

        t1 = time.perf_counter()
        p = model.predict(X_sub[te])
        infer_time_ms += (time.perf_counter() - t1) * 1000.0

        accs.append(accuracy_score(y_sub[te], p))
        precs.append(precision_score(y_sub[te], p, average='macro', zero_division=0))
        recs.append(recall_score(y_sub[te], p, average='macro', zero_division=0))
        f1s.append(f1_score(y_sub[te], p, average='macro', zero_division=0))
        mccs.append(matthews_corrcoef(y_sub[te], p))

    current, peak = tracemalloc.get_traced_memory()
    tracemalloc.stop()

    return {
        'accuracy': float(np.mean(accs)),
        'precision_macro': float(np.mean(precs)),
        'recall_macro': float(np.mean(recs)),
        'f1_macro': float(np.mean(f1s)),
        'mcc': float(np.mean(mccs)),
        'train_time_s': float(train_time_s),
        'inference_time_ms': float(infer_time_ms),
        'peak_mem_mb': float(peak) / (1024.0 * 1024.0)
    }

methods = ['GWO', 'BA', 'LightGBM', 'Boruta']

clf_factories = {
    'RandomForest': lambda s: RandomForestClassifier(
        n_estimators=800,
        max_depth=30,
        min_samples_split=3,
        min_samples_leaf=1,
        max_features='sqrt',
        class_weight='balanced',
        max_samples=0.8,
        n_jobs=-1,
        random_state=42
    ),
    'DecisionTree': lambda s: DecisionTreeClassifier(random_state=int(s), class_weight='balanced'),
    'LogisticRegression': lambda s: LogisticRegression(max_iter=1000, class_weight='balanced', random_state=int(s)),
    'GaussianNB': lambda s: GaussianNB(),
    'HistGradientBoosting': lambda s: HistGradientBoostingClassifier(random_state=int(s), learning_rate=0.1),
    'MLP': lambda s: MLPClassifier(
        random_state=int(s),
        hidden_layer_sizes=(128, 64),
        max_iter=200,
        early_stopping=True,
        n_iter_no_change=10,
        validation_fraction=0.1
    )
}

fs_cache = {}

def make_fs_base(seed: int):
    if str(FS_FITNESS_MODEL).lower() in ['rf', 'randomforest', 'random_forest']:
        return RandomForestClassifier(
            n_estimators=120,
            max_depth=18,
            min_samples_split=4,
            min_samples_leaf=2,
            max_features='sqrt',
            bootstrap=True,
            max_samples=0.7,
            class_weight='balanced_subsample',
            n_jobs=-1,
            random_state=int(seed)
        )
    return DecisionTreeClassifier(random_state=int(seed), class_weight='balanced')

def _serialize_idx(idx):
    return ','.join(str(int(i)) for i in idx)

def _mean_pairwise_jaccard(index_sets):
    sets = [set(int(i) for i in s) for s in index_sets if len(s) > 0]
    if len(sets) < 2:
        return np.nan, np.nan, 0
    vals = []
    for a, b in combinations(sets, 2):
        union = len(a | b)
        vals.append(len(a & b) / union if union else 1.0)
    return float(np.mean(vals)), float(np.std(vals)), int(len(vals))

def get_idx_and_time(seed: int, method: str, X_fs: np.ndarray, y_fs: np.ndarray):
    key = (int(seed), str(method))
    if key in fs_cache:
        return fs_cache[key]

    random.seed(int(seed))
    np.random.seed(int(seed))

    base = make_fs_base(int(seed))
    t0 = time.perf_counter()
    if str(method) == 'GWO':
        idx, _ = select_features_gwo(X_fs, y_fs, K_SUP, clf=base, epochs=FS_EPOCHS, pop_size=FS_POP_SIZE, cv=FS_CV_FOLDS, scoring=str(FS_FITNESS_SCORING), seed=int(seed))
    elif str(method) == 'BA':
        idx, _ = select_features_ba(X_fs, y_fs, K_SUP, clf=base, epochs=FS_EPOCHS, pop_size=FS_POP_SIZE, cv=FS_CV_FOLDS, scoring=str(FS_FITNESS_SCORING))
    elif str(method) == 'LightGBM':
        idx, _ = select_features_lightgbm(X_fs, y_fs, K_SUP, clf=base, cv=FS_CV_FOLDS, scoring=str(FS_FITNESS_SCORING), seed=int(seed))
    elif str(method) == 'Boruta':
        idx, _ = select_features_boruta(X_fs, y_fs, None, clf=base, cv=FS_CV_FOLDS, scoring=str(FS_FITNESS_SCORING), seed=int(seed))
    else:
        raise ValueError(f'Unknown method: {method}')

    fs_time_s = float(time.perf_counter() - t0)
    fs_cache[key] = (idx, fs_time_s)
    return idx, fs_time_s

rows = []
for seed in RUN_SEEDS:
    X_fs, y_fs = _subsample_for_fs_local(X, y, FS_MAX_SAMPLES, seed)
    for method in methods:
        idx, fs_time_s = get_idx_and_time(seed, method, X_fs, y_fs)
        X_sub = X[:, idx]
        for clf_name, make_clf in clf_factories.items():
            m = evaluate_cv_eff(make_clf(seed), X_sub, y, cv_folds=EVAL_CV_FOLDS, seed=seed)
            rows.append({
                'seed': int(seed),
                'method': str(method),
                'classifier': str(clf_name),
                'k_target': int(K_SUP) if method in ['GWO', 'BA', 'LightGBM'] else np.nan,
                'n_selected': int(len(idx)),
                'selected_features': _serialize_idx(idx),
                'fs_time_s': float(fs_time_s),
                **m
            })

fs_rows = []
for (s, m), (idx, fs_time_s) in fs_cache.items():
    fs_rows.append({
        'seed': int(s),
        'method': str(m),
        'k_target': int(K_SUP) if str(m) in ['GWO', 'BA', 'LightGBM'] else np.nan,
        'n_selected': int(len(idx)),
        'selected_features': _serialize_idx(idx),
        'fs_time_s': float(fs_time_s)
    })
fs_df = pd.DataFrame(fs_rows)
fs_df = fs_df.sort_values(['seed', 'method']).reset_index(drop=True)
display(fs_df)
display(fs_df.groupby('method', as_index=False)[['fs_time_s', 'n_selected']].agg(['mean', 'std']).reset_index())

stability_rows = []
for method_name, sub_df in fs_df.groupby('method'):
    sets = []
    for txt in sub_df['selected_features'].astype(str):
        sets.append([int(x) for x in txt.split(',') if str(x).strip() != ''])
    mean_j, std_j, n_pairs = _mean_pairwise_jaccard(sets)
    stability_rows.append({
        'method': str(method_name),
        'mean_pairwise_jaccard': mean_j,
        'std_pairwise_jaccard': std_j,
        'n_pairs': n_pairs
    })
stability_df = pd.DataFrame(stability_rows).sort_values('mean_pairwise_jaccard', ascending=False).reset_index(drop=True)
print('Feature-selection stability across runs')
display(stability_df)

raw_df = pd.DataFrame(rows)
display(raw_df)
print('Rows:', len(raw_df))

metrics_cols = ['accuracy', 'precision_macro', 'recall_macro', 'f1_macro', 'mcc', 'fs_time_s', 'train_time_s', 'inference_time_ms', 'peak_mem_mb', 'n_selected']

means = raw_df.groupby(['method', 'classifier'])[metrics_cols].mean().reset_index()
means = means.rename(columns={
    'method': 'Algorithm',
    'classifier': 'Classifier',
    'accuracy': 'Accuracy',
    'precision_macro': 'Precision',
    'recall_macro': 'Recall',
    'f1_macro': 'F1_macro',
    'mcc': 'MCC',
    'n_selected': 'N_Selected'
})

eval_df = means[['Algorithm', 'Classifier', 'Accuracy', 'Precision', 'Recall', 'F1_macro', 'MCC', 'N_Selected', 'fs_time_s', 'train_time_s', 'inference_time_ms', 'peak_mem_mb']].copy()
eval_df = eval_df.sort_values(['F1_macro', 'MCC', 'Accuracy'], ascending=[False, False, False]).reset_index(drop=True)
display(eval_df)

print('Deployment-oriented summary (lower downstream cost is better)')
deployment_df = eval_df[['Algorithm', 'Classifier', 'N_Selected', 'train_time_s', 'inference_time_ms', 'peak_mem_mb']].copy()
deployment_df = deployment_df.sort_values(['N_Selected', 'peak_mem_mb', 'inference_time_ms', 'train_time_s'], ascending=[True, True, True, True]).reset_index(drop=True)
display(deployment_df)

agg = raw_df.groupby(['method', 'classifier'])[metrics_cols].agg(['mean', 'std'])
agg.columns = [f"{m}_{s}" for (m, s) in agg.columns]
agg = agg.reset_index()

def fmt_pair(m, s, is_time=False):
    if pd.isna(m):
        return ''
    if pd.isna(s):
        return f"{m:.4f}" if not is_time else f"{m:.3f}"
    return (f"{m:.4f} +- {s:.4f}" if not is_time else f"{m:.3f} +- {s:.3f}")

final_table = pd.DataFrame({
    'Method': agg['method'].astype(str),
    'Classifier': agg['classifier'].astype(str),
})

for src, dst in {
    'accuracy': 'accuracy',
    'precision_macro': 'precision_macro',
    'recall_macro': 'recall_macro',
    'f1_macro': 'f1_macro',
    'mcc': 'mcc',
    'n_selected': 'n_selected',
    'fs_time_s': 'fs_time_s',
    'train_time_s': 'train_time_s',
    'inference_time_ms': 'inference_time_ms',
    'peak_mem_mb': 'peak_mem_mb'
}.items():
    mcol = f"{src}_mean"
    scol = f"{src}_std"
    is_time = src in ['fs_time_s', 'train_time_s', 'inference_time_ms', 'peak_mem_mb']
    final_table[dst] = [fmt_pair(m, s, is_time=is_time) for m, s in zip(agg[mcol], agg[scol])]

final_table = final_table.sort_values(['Classifier', 'Method']).reset_index(drop=True)
display(final_table)

2026/06/14 01:47:16 PM, INFO, mealpy.swarm_based.GWO.OriginalGWO: OriginalGWO(epoch=30, pop_size=10)


2026/06/14 01:47:26 PM, INFO, mealpy.swarm_based.GWO.OriginalGWO: >>>Problem: P, Epoch: 1, Current best: 0.8147768967203118, Global best: 0.8147768967203118, Runtime: 5.65913 seconds


2026/06/14 01:47:31 PM, INFO, mealpy.swarm_based.GWO.OriginalGWO: >>>Problem: P, Epoch: 2, Current best: 0.8165297724646787, Global best: 0.8165297724646787, Runtime: 5.02645 seconds


2026/06/14 01:47:37 PM, INFO, mealpy.swarm_based.GWO.OriginalGWO: >>>Problem: P, Epoch: 3, Current best: 0.8280529794639805, Global best: 0.8280529794639805, Runtime: 5.74369 seconds


2026/06/14 01:47:43 PM, INFO, mealpy.swarm_based.GWO.OriginalGWO: >>>Problem: P, Epoch: 4, Current best: 0.8343491493558525, Global best: 0.8343491493558525, Runtime: 5.83063 seconds


...

Rows: 240


,Algorithm,Classifier,Accuracy,Precision,Recall,F1_macro,MCC,N_Selected,fs_time_s,train_time_s,inference_time_ms,peak_mem_mb
0,GWO,HistGradientBoosting,0.906596,0.894120,0.896620,0.880769,0.897344,10.0,145.713449,218.364505,3068.601717,31.914554
1,Boruta,HistGradientBoosting,0.906479,0.926405,0.858542,0.875071,0.897147,9.6,8.746015,218.603493,5141.179606,31.286489
2,Boruta,RandomForest,0.900429,0.881602,0.904416,0.873506,0.890956,9.6,8.746015,86.508932,8924.850808,39.110456
3,GWO,RandomForest,0.897434,0.862371,0.900914,0.864774,0.887245,10.0,145.713449,87.933528,9308.927211,39.444783
4,Boruta,DecisionTree,0.885370,0.867964,0.891134,0.860484,0.874217,9.6,8.746015,1.059072,29.660148,24.223118
5,Boruta,MLP,0.886428,0.910627,0.841907,0.858498,0.875043,9.6,8.746015,190.705464,202.314121,46.898514
6,GWO,MLP,0.884424,0.891326,0.857879,0.858363,0.872747,10.0,145.713449,185.892968,187.472076,47.079790
7,GWO,DecisionTree,0.884637,0.850973,0.889785,0.853773,0.873048,10.0,145.713449,0.915479,39.436614,25.134440
8,BA,HistGradientBoosting,0.868912,0.860114,0.850433,0.836850,0.856344,10.0,182.031650,205.218270,3702.343644,31.874806
9,LightGBM,HistGradientBoosting,0.900912,0.877017,0.860388,0.836775,0.891164,10.0,114.733429,226.095444,4789.415971,31.942298


Deployment-oriented summary (lower downstream cost is better)


,Algorithm,Classifier,N_Selected,train_time_s,inference_time_ms,peak_mem_mb
0,Boruta,GaussianNB,9.6,0.284978,94.045555,24.217820
1,Boruta,DecisionTree,9.6,1.059072,29.660148,24.223118
2,Boruta,LogisticRegression,9.6,26.980971,86.775896,24.264647
3,Boruta,HistGradientBoosting,9.6,218.603493,5141.179606,31.286489
4,Boruta,RandomForest,9.6,86.508932,8924.850808,39.110456
5,Boruta,MLP,9.6,190.705464,202.314121,46.898514
6,GWO,GaussianNB,10.0,0.279819,84.824871,25.130383
7,LightGBM,GaussianNB,10.0,0.282074,92.525154,25.130445
8,BA,GaussianNB,10.0,0.294250,81.490792,25.130653
9,GWO,DecisionTree,10.0,0.915479,39.436614,25.134440


,Method,Classifier,accuracy,precision_macro,recall_macro,f1_macro,mcc,n_selected,fs_time_s,train_time_s,inference_time_ms,peak_mem_mb
0,BA,DecisionTree,0.8490 +- 0.0229,0.8147 +- 0.0253,0.8525 +- 0.0248,0.8175 +- 0.0244,0.8339 +- 0.0251,10.0000 +- 0.0000,182.032 +- 20.630,0.980 +- 0.320,30.918 +- 2.051,25.135 +- 0.002
1,Boruta,DecisionTree,0.8854 +- 0.0010,0.8680 +- 0.0009,0.8911 +- 0.0009,0.8605 +- 0.0009,0.8742 +- 0.0011,9.6000 +- 0.6992,8.746 +- 1.408,1.059 +- 0.180,29.660 +- 5.419,24.223 +- 1.595
2,GWO,DecisionTree,0.8846 +- 0.0041,0.8510 +- 0.0116,0.8898 +- 0.0088,0.8538 +- 0.0077,0.8730 +- 0.0046,10.0000 +- 0.0000,145.713 +- 17.088,0.915 +- 0.348,39.437 +- 23.458,25.134 +- 0.001
3,LightGBM,DecisionTree,0.8816 +- 0.0012,0.8170 +- 0.0081,0.8463 +- 0.0017,0.8145 +- 0.0016,0.8697 +- 0.0013,10.0000 +- 0.0000,114.733 +- 26.985,1.117 +- 0.067,29.918 +- 3.751,25.135 +- 0.003
4,BA,GaussianNB,0.2381 +- 0.0794,0.2749 +- 0.0850,0.3127 +- 0.0761,0.2251 +- 0.0777,0.1899 +- 0.0854,10.0000 +- 0.0000,182.032 +- 20.630,0.294 +- 0.041,81.491 +- 4.797,25.131 +- 0.003
5,Boruta,GaussianNB,0.4096 +- 0.0058,0.3493 +- 0.0294,0.4766 +- 0.0050,0.3481 +- 0.0110,0.3695 +- 0.0062,9.6000 +- 0.6992,8.746 +- 1.408,0.285 +- 0.037,94.046 +- 14.409,24.218 +- 1.595
6,GWO,GaussianNB,0.3090 +- 0.0473,0.3119 +- 0.0232,0.3803 +- 0.0434,0.2885 +- 0.0324,0.2699 +- 0.0494,10.0000 +- 0.0000,145.713 +- 17.088,0.280 +- 0.031,84.825 +- 5.755,25.130 +- 0.001
7,LightGBM,GaussianNB,0.3371 +- 0.0052,0.2438 +- 0.0025,0.3790 +- 0.0046,0.2701 +- 0.0033,0.2993 +- 0.0046,10.0000 +- 0.0000,114.733 +- 26.985,0.282 +- 0.022,92.525 +- 15.547,25.130 +- 0.002
8,BA,HistGradientBoosting,0.8689 +- 0.0287,0.8601 +- 0.0386,0.8504 +- 0.0452,0.8368 +- 0.0379,0.8563 +- 0.0307,10.0000 +- 0.0000,182.032 +- 20.630,205.218 +- 63.941,3702.344 +- 1863.182,31.875 +- 0.366
9,Boruta,HistGradientBoosting,0.9065 +- 0.0011,0.9264 +- 0.0023,0.8585 +- 0.0022,0.8751 +- 0.0020,0.8971 +- 0.0012,9.6000 +- 0.6992,8.746 +- 1.408,218.603 +- 61.729,5141.180 +- 2406.883,31.286 +- 0.890


## Statistical Significance and Pairwise Tests

In [5]:
try:
    from scipy.stats import friedmanchisquare, wilcoxon
    _HAS_SCIPY = True
except Exception:
    _HAS_SCIPY = False

def _prepare_pivot(df: pd.DataFrame, value_col: str, ordered_methods):
    pivot = df.pivot_table(index='seed', columns='method', values=value_col, aggfunc='mean')
    pivot = pivot.dropna(axis=1, how='all')
    ordered = [m for m in list(ordered_methods) if m in pivot.columns]
    pivot = pivot[ordered]
    pivot = pivot.dropna(axis=0, how='any')
    return pivot

def _holm_adjust(pvals):
    if len(pvals) == 0:
        return []
    order = np.argsort(pvals)
    ranked = np.asarray(pvals, dtype=float)[order]
    m = len(ranked)
    adj = np.empty(m, dtype=float)
    running = 0.0
    for i, p in enumerate(ranked):
        value = min(1.0, (m - i) * float(p))
        running = max(running, value)
        adj[i] = running
    restored = np.empty(m, dtype=float)
    restored[order] = adj
    return restored.tolist()

def _safe_wilcoxon(a, b):
    if (not _HAS_SCIPY) or (len(a) < 2):
        return None
    if np.allclose(np.asarray(a) - np.asarray(b), 0.0):
        return 1.0
    return float(wilcoxon(a, b, zero_method='wilcox').pvalue)

def global_significance_for_classifier(df: pd.DataFrame, clf_name: str, methods_list, metric: str = 'f1_macro'):
    sub = df[df['classifier'].astype(str) == str(clf_name)].copy()
    if sub.empty:
        return {'Classifier': str(clf_name), 'metric': metric, 'test': None, 'p_value': None, 'reason': 'no rows'}
    pivot = _prepare_pivot(sub, metric, methods_list)
    methods_avail = list(pivot.columns)
    n_runs = int(pivot.shape[0])
    if len(methods_avail) < 2 or n_runs < 2:
        return {'Classifier': str(clf_name), 'metric': metric, 'test': None, 'p_value': None, 'methods': methods_avail, 'n_runs': n_runs, 'reason': 'need >=2 methods with complete runs'}
    if not _HAS_SCIPY:
        return {'Classifier': str(clf_name), 'metric': metric, 'test': None, 'p_value': None, 'methods': methods_avail, 'n_runs': n_runs, 'reason': 'scipy not available'}
    if len(methods_avail) == 2:
        a, b = methods_avail
        p = _safe_wilcoxon(pivot[a].values, pivot[b].values)
        return {'Classifier': str(clf_name), 'metric': metric, 'test': 'Wilcoxon', 'p_value': p, 'methods': methods_avail, 'n_runs': n_runs}
    samples = [pivot[m].values for m in methods_avail]
    _, p = friedmanchisquare(*samples)
    return {'Classifier': str(clf_name), 'metric': metric, 'test': 'Friedman', 'p_value': float(p), 'methods': methods_avail, 'n_runs': n_runs}

def pairwise_vs_reference_for_classifier(df: pd.DataFrame, clf_name: str, methods_list, reference: str = 'GWO', metric: str = 'f1_macro'):
    sub = df[df['classifier'].astype(str) == str(clf_name)].copy()
    if sub.empty:
        return []
    pivot = _prepare_pivot(sub, metric, methods_list)
    if reference not in pivot.columns:
        return []
    rows = []
    pvals = []
    valid_idx = []
    for method in pivot.columns:
        if str(method) == str(reference):
            continue
        pair = pivot[[reference, method]].dropna(axis=0, how='any')
        n_runs = int(pair.shape[0])
        row = {
            'Classifier': str(clf_name),
            'metric': metric,
            'reference': str(reference),
            'comparison': str(method),
            'n_runs': n_runs,
            'mean_ref': float(pair[reference].mean()) if n_runs else np.nan,
            'mean_cmp': float(pair[method].mean()) if n_runs else np.nan,
            'wins_ref': int((pair[reference] > pair[method]).sum()) if n_runs else 0,
            'ties': int((pair[reference] == pair[method]).sum()) if n_runs else 0,
            'wins_cmp': int((pair[reference] < pair[method]).sum()) if n_runs else 0,
            'test': 'Wilcoxon',
            'p_value': None,
            'p_holm': None,
            'better_by_mean': str(reference) if (n_runs and pair[reference].mean() >= pair[method].mean()) else str(method)
        }
        if _HAS_SCIPY and n_runs >= 2:
            p = _safe_wilcoxon(pair[reference].values, pair[method].values)
            row['p_value'] = p
            if p is not None:
                valid_idx.append(len(rows))
                pvals.append(p)
        else:
            row['test'] = None
        rows.append(row)
    adj = _holm_adjust(pvals)
    for idx, p_adj in zip(valid_idx, adj):
        rows[idx]['p_holm'] = float(p_adj)
    return rows

def global_significance_for_fs(fs_df_local: pd.DataFrame, methods_list, metric: str = 'fs_time_s'):
    pivot = _prepare_pivot(fs_df_local.copy(), metric, methods_list)
    methods_avail = list(pivot.columns)
    n_runs = int(pivot.shape[0])
    if len(methods_avail) < 2 or n_runs < 2:
        return {'metric': metric, 'test': None, 'p_value': None, 'methods': methods_avail, 'n_runs': n_runs, 'reason': 'need >=2 methods with complete runs'}
    if not _HAS_SCIPY:
        return {'metric': metric, 'test': None, 'p_value': None, 'methods': methods_avail, 'n_runs': n_runs, 'reason': 'scipy not available'}
    if len(methods_avail) == 2:
        a, b = methods_avail
        p = _safe_wilcoxon(pivot[a].values, pivot[b].values)
        return {'metric': metric, 'test': 'Wilcoxon', 'p_value': p, 'methods': methods_avail, 'n_runs': n_runs}
    samples = [pivot[m].values for m in methods_avail]
    _, p = friedmanchisquare(*samples)
    return {'metric': metric, 'test': 'Friedman', 'p_value': float(p), 'methods': methods_avail, 'n_runs': n_runs}

def pairwise_vs_reference_for_fs(fs_df_local: pd.DataFrame, methods_list, reference: str = 'GWO', metric: str = 'fs_time_s'):
    pivot = _prepare_pivot(fs_df_local.copy(), metric, methods_list)
    if reference not in pivot.columns:
        return []
    rows = []
    pvals = []
    valid_idx = []
    for method in pivot.columns:
        if str(method) == str(reference):
            continue
        pair = pivot[[reference, method]].dropna(axis=0, how='any')
        n_runs = int(pair.shape[0])
        mean_ref = float(pair[reference].mean()) if n_runs else np.nan
        mean_cmp = float(pair[method].mean()) if n_runs else np.nan
        row = {
            'metric': metric,
            'reference': str(reference),
            'comparison': str(method),
            'n_runs': n_runs,
            'mean_ref': mean_ref,
            'mean_cmp': mean_cmp,
            'wins_ref': int((pair[reference] < pair[method]).sum()) if n_runs else 0,
            'ties': int((pair[reference] == pair[method]).sum()) if n_runs else 0,
            'wins_cmp': int((pair[reference] > pair[method]).sum()) if n_runs else 0,
            'test': 'Wilcoxon',
            'p_value': None,
            'p_holm': None,
            'faster_by_mean': str(reference) if (n_runs and mean_ref <= mean_cmp) else str(method)
        }
        if _HAS_SCIPY and n_runs >= 2:
            p = _safe_wilcoxon(pair[reference].values, pair[method].values)
            row['p_value'] = p
            if p is not None:
                valid_idx.append(len(rows))
                pvals.append(p)
        else:
            row['test'] = None
        rows.append(row)
    adj = _holm_adjust(pvals)
    for idx, p_adj in zip(valid_idx, adj):
        rows[idx]['p_holm'] = float(p_adj)
    return rows

print('Global significance for performance (f1_macro)')
sig_rows = []
for clf_name in sorted(raw_df['classifier'].astype(str).unique()):
    sig_rows.append(global_significance_for_classifier(raw_df, clf_name, methods, metric='f1_macro'))
sig_df = pd.DataFrame(sig_rows)
display(sig_df)

print('Pairwise post-hoc vs GWO for performance (f1_macro)')
pairwise_rows = []
for clf_name in sorted(raw_df['classifier'].astype(str).unique()):
    pairwise_rows.extend(pairwise_vs_reference_for_classifier(raw_df, clf_name, methods, reference='GWO', metric='f1_macro'))
pairwise_df = pd.DataFrame(pairwise_rows)
display(pairwise_df)

print('Global significance for feature-selection time (fs_time_s)')
sig_df_time = pd.DataFrame([global_significance_for_fs(fs_df, methods, metric='fs_time_s')])
display(sig_df_time)

print('Pairwise post-hoc vs GWO for feature-selection time (fs_time_s)')
pairwise_time_df = pd.DataFrame(pairwise_vs_reference_for_fs(fs_df, methods, reference='GWO', metric='fs_time_s'))
display(pairwise_time_df)

print('Global significance for number of selected features (n_selected)')
sig_df_n_selected = pd.DataFrame([global_significance_for_fs(fs_df, methods, metric='n_selected')])
display(sig_df_n_selected)

print('Pairwise post-hoc vs GWO for number of selected features (n_selected)')
pairwise_n_selected_df = pd.DataFrame(pairwise_vs_reference_for_fs(fs_df, methods, reference='GWO', metric='n_selected'))
display(pairwise_n_selected_df)

print('Stability summary (mean pairwise Jaccard across runs)')
display(stability_df)

Global significance for performance (f1_macro)


,Classifier,metric,test,p_value,methods,n_runs
0,DecisionTree,f1_macro,Friedman,0.000020,"[GWO, BA, LightGBM, Boruta]",10
1,GaussianNB,f1_macro,Friedman,0.000295,"[GWO, BA, LightGBM, Boruta]",10
2,HistGradientBoosting,f1_macro,Friedman,0.000033,"[GWO, BA, LightGBM, Boruta]",10
3,LogisticRegression,f1_macro,Friedman,0.000187,"[GWO, BA, LightGBM, Boruta]",10
4,MLP,f1_macro,Friedman,0.000024,"[GWO, BA, LightGBM, Boruta]",10
5,RandomForest,f1_macro,Friedman,0.000050,"[GWO, BA, LightGBM, Boruta]",10


Pairwise post-hoc vs GWO for performance (f1_macro)


,Classifier,metric,reference,comparison,n_runs,mean_ref,mean_cmp,wins_ref,ties,wins_cmp,test,p_value,p_holm,better_by_mean
0,DecisionTree,f1_macro,GWO,BA,10,0.853773,0.817526,10,0,0,Wilcoxon,0.001953,0.005859,GWO
1,DecisionTree,f1_macro,GWO,LightGBM,10,0.853773,0.814544,10,0,0,Wilcoxon,0.001953,0.005859,GWO
2,DecisionTree,f1_macro,GWO,Boruta,10,0.853773,0.860484,3,0,7,Wilcoxon,0.048828,0.048828,Boruta
3,GaussianNB,f1_macro,GWO,BA,10,0.288460,0.225077,8,0,2,Wilcoxon,0.064453,0.128906,GWO
4,GaussianNB,f1_macro,GWO,LightGBM,10,0.288460,0.270112,8,0,2,Wilcoxon,0.105469,0.128906,GWO
5,GaussianNB,f1_macro,GWO,Boruta,10,0.288460,0.348131,0,0,10,Wilcoxon,0.001953,0.005859,Boruta
6,HistGradientBoosting,f1_macro,GWO,BA,10,0.880769,0.836850,10,0,0,Wilcoxon,0.001953,0.005859,GWO
7,HistGradientBoosting,f1_macro,GWO,LightGBM,10,0.880769,0.836775,10,0,0,Wilcoxon,0.001953,0.005859,GWO
8,HistGradientBoosting,f1_macro,GWO,Boruta,10,0.880769,0.875071,8,0,2,Wilcoxon,0.037109,0.037109,GWO
9,LogisticRegression,f1_macro,GWO,BA,10,0.561587,0.436126,9,0,1,Wilcoxon,0.003906,0.007812,GWO


Global significance for feature-selection time (fs_time_s)


...

,metric,test,p_value,methods,n_runs
0,n_selected,Friedman,0.029291,"[GWO, BA, LightGBM, Boruta]",10


Pairwise post-hoc vs GWO for number of selected features (n_selected)


,metric,reference,comparison,n_runs,mean_ref,mean_cmp,wins_ref,ties,wins_cmp,test,p_value,p_holm,faster_by_mean
0,n_selected,GWO,BA,10,10.0,10.0,0,10,0,Wilcoxon,1.00,1.00,GWO
1,n_selected,GWO,LightGBM,10,10.0,10.0,0,10,0,Wilcoxon,1.00,1.00,GWO
2,n_selected,GWO,Boruta,10,10.0,9.6,0,7,3,Wilcoxon,0.25,0.75,Boruta


Stability summary (mean pairwise Jaccard across runs)


,method,mean_pairwise_jaccard,std_pairwise_jaccard,n_pairs
0,LightGBM,1.000000,0.000000,45
1,Boruta,0.932840,0.073291,45
2,GWO,0.585193,0.100605,45
3,BA,0.360625,0.100359,45


## Without FS 


In [ ]:
import random
import time
import tracemalloc
from itertools import combinations
from sklearn.base import clone
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, f1_score, matthews_corrcoef, precision_score, recall_score
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier

pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)
pd.set_option('display.width', None)

N_RUNS = 10
RUN_SEEDS = list(range(N_RUNS))
K_SUP = 10
N_FEATURES_TOTAL = int(X.shape[1])

FS_MAX_SAMPLES = 50000
FS_EPOCHS = 30
FS_POP_SIZE = 10
FS_CV_FOLDS = 3

FS_FITNESS_MODEL = 'DecisionTree'
FS_FITNESS_SCORING = 'f1_macro'

EVAL_CV_FOLDS = 3

def _subsample_for_fs_local(X_in: np.ndarray, y_in: np.ndarray, max_samples: int, seed: int):
    if (max_samples is None) or (X_in.shape[0] <= int(max_samples)):
        return X_in, y_in
    rng = np.random.default_rng(int(seed))
    idx = rng.choice(X_in.shape[0], size=int(max_samples), replace=False)
    return X_in[idx], y_in[idx]

def evaluate_cv_eff(clf, X_sub: np.ndarray, y_sub: np.ndarray, cv_folds: int, seed: int):
    cvk = StratifiedKFold(n_splits=int(cv_folds), shuffle=True, random_state=int(seed))

    accs, precs, recs, f1s, mccs = [], [], [], [], []
    train_time_s = 0.0
    infer_time_ms = 0.0

    tracemalloc.start()
    for tr, te in cvk.split(X_sub, y_sub):
        model = Pipeline([
            ('imputer', SimpleImputer(strategy='median')),
            ('scaler', StandardScaler(with_mean=True)),
            ('clf', clone(clf))
        ])

        t0 = time.perf_counter()
        model.fit(X_sub[tr], y_sub[tr])
        train_time_s += (time.perf_counter() - t0)

        t1 = time.perf_counter()
        p = model.predict(X_sub[te])
        infer_time_ms += (time.perf_counter() - t1) * 1000.0

        accs.append(accuracy_score(y_sub[te], p))
        precs.append(precision_score(y_sub[te], p, average='macro', zero_division=0))
        recs.append(recall_score(y_sub[te], p, average='macro', zero_division=0))
        f1s.append(f1_score(y_sub[te], p, average='macro', zero_division=0))
        mccs.append(matthews_corrcoef(y_sub[te], p))

    current, peak = tracemalloc.get_traced_memory()
    tracemalloc.stop()

    return {
        'accuracy': float(np.mean(accs)),
        'precision_macro': float(np.mean(precs)),
        'recall_macro': float(np.mean(recs)),
        'f1_macro': float(np.mean(f1s)),
        'mcc': float(np.mean(mccs)),
        'train_time_s': float(train_time_s),
        'inference_time_ms': float(infer_time_ms),
        'peak_mem_mb': float(peak) / (1024.0 * 1024.0)
    }

methods = ['Without FS']

clf_factories = {
    'RandomForest': lambda s: RandomForestClassifier(
        n_estimators=800,
        max_depth=30,
        min_samples_split=3,
        min_samples_leaf=1,
        max_features='sqrt',
        class_weight='balanced',
        max_samples=0.8,
        n_jobs=-1,
        random_state=42
    ),
    'DecisionTree': lambda s: DecisionTreeClassifier(random_state=int(s), class_weight='balanced'),
    'LogisticRegression': lambda s: LogisticRegression(max_iter=1000, class_weight='balanced', random_state=int(s)),
    'GaussianNB': lambda s: GaussianNB(),
    'HistGradientBoosting': lambda s: HistGradientBoostingClassifier(random_state=int(s), learning_rate=0.1),
    'MLP': lambda s: MLPClassifier(
        random_state=int(s),
        hidden_layer_sizes=(128, 64),
        max_iter=200,
        early_stopping=True,
        n_iter_no_change=10,
        validation_fraction=0.1
    )
}

fs_cache = {}

def make_fs_base(seed: int):
    if str(FS_FITNESS_MODEL).lower() in ['rf', 'randomforest', 'random_forest']:
        return RandomForestClassifier(
            n_estimators=120,
            max_depth=18,
            min_samples_split=4,
            min_samples_leaf=2,
            max_features='sqrt',
            bootstrap=True,
            max_samples=0.7,
            class_weight='balanced_subsample',
            n_jobs=-1,
            random_state=int(seed)
        )
    return DecisionTreeClassifier(random_state=int(seed), class_weight='balanced')

def _serialize_idx(idx):
    return ','.join(str(int(i)) for i in idx)

def _mean_pairwise_jaccard(index_sets):
    sets = [set(int(i) for i in s) for s in index_sets if len(s) > 0]
    if len(sets) < 2:
        return np.nan, np.nan, 0
    vals = []
    for a, b in combinations(sets, 2):
        union = len(a | b)
        vals.append(len(a & b) / union if union else 1.0)
    return float(np.mean(vals)), float(np.std(vals)), int(len(vals))

def get_idx_and_time(seed: int, method: str, X_fs: np.ndarray, y_fs: np.ndarray):
    key = (int(seed), str(method))
    if key in fs_cache:
        return fs_cache[key]

    random.seed(int(seed))
    np.random.seed(int(seed))

    base = make_fs_base(int(seed))
    t0 = time.perf_counter()
    if str(method) == 'Without FS':
        idx, _ = select_features_without_fs(X_fs, y_fs, clf=base, cv=FS_CV_FOLDS, scoring=str(FS_FITNESS_SCORING))
    else:
        raise ValueError(f'Unknown method: {method}')

    fs_time_s = float(time.perf_counter() - t0)
    fs_cache[key] = (idx, fs_time_s)
    return idx, fs_time_s

rows = []
for seed in RUN_SEEDS:
    X_fs, y_fs = _subsample_for_fs_local(X, y, FS_MAX_SAMPLES, seed)
    for method in methods:
        idx, fs_time_s = get_idx_and_time(seed, method, X_fs, y_fs)
        X_sub = X[:, idx]
        for clf_name, make_clf in clf_factories.items():
            m = evaluate_cv_eff(make_clf(seed), X_sub, y, cv_folds=EVAL_CV_FOLDS, seed=seed)
            rows.append({
                'seed': int(seed),
                'method': str(method),
                'classifier': str(clf_name),
                'k_target': int(K_SUP) if method in [] else np.nan,
                'n_selected': int(len(idx)),
                'selected_features': _serialize_idx(idx),
                'fs_time_s': float(fs_time_s),
                **m
            })

fs_rows = []
for (s, m), (idx, fs_time_s) in fs_cache.items():
    fs_rows.append({
        'seed': int(s),
        'method': str(m),
        'k_target': int(K_SUP) if str(m) in [] else np.nan,
        'n_selected': int(len(idx)),
        'selected_features': _serialize_idx(idx),
        'fs_time_s': float(fs_time_s)
    })
fs_df = pd.DataFrame(fs_rows)
fs_df = fs_df.sort_values(['seed', 'method']).reset_index(drop=True)
display(fs_df)
display(fs_df.groupby('method', as_index=False)[['fs_time_s', 'n_selected']].agg(['mean', 'std']).reset_index())

stability_rows = []
for method_name, sub_df in fs_df.groupby('method'):
    sets = []
    for txt in sub_df['selected_features'].astype(str):
        sets.append([int(x) for x in txt.split(',') if str(x).strip() != ''])
    mean_j, std_j, n_pairs = _mean_pairwise_jaccard(sets)
    stability_rows.append({
        'method': str(method_name),
        'mean_pairwise_jaccard': mean_j,
        'std_pairwise_jaccard': std_j,
        'n_pairs': n_pairs
    })
stability_df = pd.DataFrame(stability_rows).sort_values('mean_pairwise_jaccard', ascending=False).reset_index(drop=True)
print('Feature-selection stability across runs')
display(stability_df)

raw_df = pd.DataFrame(rows)
display(raw_df)
print('Rows:', len(raw_df))

metrics_cols = ['accuracy', 'precision_macro', 'recall_macro', 'f1_macro', 'mcc', 'fs_time_s', 'train_time_s', 'inference_time_ms', 'peak_mem_mb', 'n_selected']

means = raw_df.groupby(['method', 'classifier'])[metrics_cols].mean().reset_index()
means = means.rename(columns={
    'method': 'Algorithm',
    'classifier': 'Classifier',
    'accuracy': 'Accuracy',
    'precision_macro': 'Precision',
    'recall_macro': 'Recall',
    'f1_macro': 'F1_macro',
    'mcc': 'MCC',
    'n_selected': 'N_Selected'
})

eval_df = means[['Algorithm', 'Classifier', 'Accuracy', 'Precision', 'Recall', 'F1_macro', 'MCC', 'N_Selected', 'fs_time_s', 'train_time_s', 'inference_time_ms', 'peak_mem_mb']].copy()
eval_df = eval_df.sort_values(['F1_macro', 'MCC', 'Accuracy'], ascending=[False, False, False]).reset_index(drop=True)
display(eval_df)

print('Deployment-oriented summary (lower downstream cost is better)')
deployment_df = eval_df[['Algorithm', 'Classifier', 'N_Selected', 'train_time_s', 'inference_time_ms', 'peak_mem_mb']].copy()
deployment_df = deployment_df.sort_values(['N_Selected', 'peak_mem_mb', 'inference_time_ms', 'train_time_s'], ascending=[True, True, True, True]).reset_index(drop=True)
display(deployment_df)

agg = raw_df.groupby(['method', 'classifier'])[metrics_cols].agg(['mean', 'std'])
agg.columns = [f"{m}_{s}" for (m, s) in agg.columns]
agg = agg.reset_index()

def fmt_pair(m, s, is_time=False):
    if pd.isna(m):
        return ''
    if pd.isna(s):
        return f"{m:.4f}" if not is_time else f"{m:.3f}"
    return (f"{m:.4f} +- {s:.4f}" if not is_time else f"{m:.3f} +- {s:.3f}")

final_table = pd.DataFrame({
    'Method': agg['method'].astype(str),
    'Classifier': agg['classifier'].astype(str),
})

for src, dst in {
    'accuracy': 'accuracy',
    'precision_macro': 'precision_macro',
    'recall_macro': 'recall_macro',
    'f1_macro': 'f1_macro',
    'mcc': 'mcc',
    'n_selected': 'n_selected',
    'fs_time_s': 'fs_time_s',
    'train_time_s': 'train_time_s',
    'inference_time_ms': 'inference_time_ms',
    'peak_mem_mb': 'peak_mem_mb'
}.items():
    mcol = f"{src}_mean"
    scol = f"{src}_std"
    is_time = src in ['fs_time_s', 'train_time_s', 'inference_time_ms', 'peak_mem_mb']
    final_table[dst] = [fmt_pair(m, s, is_time=is_time) for m, s in zip(agg[mcol], agg[scol])]

final_table = final_table.sort_values(['Classifier', 'Method']).reset_index(drop=True)
display(final_table)

,seed,method,k_target,n_selected,selected_features,fs_time_s
0,0,Without FS,NaN,24,"0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23",1.063728
1,1,Without FS,NaN,24,"0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23",0.824218
2,2,Without FS,NaN,24,"0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23",0.829466
3,3,Without FS,NaN,24,"0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23",0.868167
4,4,Without FS,NaN,24,"0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23",0.955720
5,5,Without FS,NaN,24,"0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23",0.831754
6,6,Without FS,NaN,24,"0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23",0.879789
7,7,Without FS,NaN,24,"0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23",0.849581
8,8,Without FS,NaN,24,"0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23",0.910403
9,9,Without FS,NaN,24,"0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23",0.940571


index      method fs_time_s           n_selected     
                         mean       std       mean  std
0     0  Without FS   0.89534  0.075121       24.0  0.0

Feature-selection stability across runs


,method,mean_pairwise_jaccard,std_pairwise_jaccard,n_pairs
0,Without FS,1.0,0.0,45


,seed,method,classifier,k_target,n_selected,selected_features,fs_time_s,accuracy,precision_macro,recall_macro,f1_macro,mcc,train_time_s,inference_time_ms,peak_mem_mb
0,0,Without FS,RandomForest,NaN,24,"0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23",1.063728,0.906542,0.865420,0.911031,0.874474,0.897188,86.835590,8231.948087,57.149384
1,0,Without FS,DecisionTree,NaN,24,"0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23",1.063728,0.891802,0.851988,0.898156,0.861778,0.880793,1.866413,55.395085,57.076730
2,0,Without FS,LogisticRegression,NaN,24,"0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23",1.063728,0.683807,0.649636,0.719981,0.666868,0.652384,68.531575,127.975835,57.076025
3,0,Without FS,GaussianNB,NaN,24,"0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23",1.063728,0.236240,0.288249,0.318651,0.239624,0.180903,0.709744,143.007287,57.069308
4,0,Without FS,HistGradientBoosting,NaN,24,"0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23",1.063728,0.905020,0.882390,0.906516,0.881641,0.895524,224.778933,4825.372909,57.138994
5,0,Without FS,MLP,NaN,24,"0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23",1.063728,0.890760,0.886985,0.879658,0.868961,0.879861,195.720208,271.333835,57.078880
6,1,Without FS,RandomForest,NaN,24,"0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23",0.824218,0.906038,0.855208,0.912110,0.869122,0.896673,76.201139,7910.878290,57.122377
7,1,Without FS,DecisionTree,NaN,24,"0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23",0.824218,0.891670,0.841971,0.899505,0.856719,0.880677,1.601930,61.623360,57.078476
8,1,Without FS,LogisticRegression,NaN,24,"0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23",0.824218,0.683052,0.642381,0.720726,0.662284,0.651627,75.215775,100.618156,57.075308
9,1,Without FS,GaussianNB,NaN,24,"0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23",0.824218,0.231663,0.304126,0.314701,0.235643,0.175140,0.535495,147.161208,57.070086


Rows: 60


,Algorithm,Classifier,Accuracy,Precision,Recall,F1_macro,MCC,N_Selected,fs_time_s,train_time_s,inference_time_ms,peak_mem_mb
0,Without FS,HistGradientBoosting,0.908212,0.885422,0.910195,0.885186,0.899067,24.0,0.89534,205.572521,4068.767411,57.136985
1,Without FS,MLP,0.891710,0.884603,0.883689,0.870174,0.880858,24.0,0.89534,193.949444,180.701969,57.084051
2,Without FS,RandomForest,0.905183,0.855368,0.911207,0.868249,0.895735,24.0,0.89534,81.879798,8703.595383,57.134888
3,Without FS,DecisionTree,0.890554,0.842005,0.898411,0.855633,0.879462,24.0,0.89534,1.645224,52.736277,57.077119
4,Without FS,LogisticRegression,0.682887,0.643190,0.720439,0.662316,0.651462,24.0,0.89534,67.820292,98.193237,57.076390
5,Without FS,GaussianNB,0.231713,0.290480,0.314582,0.234623,0.175081,24.0,0.89534,0.569870,150.931727,57.070244


Deployment-oriented summary (lower downstream cost is better)


,Algorithm,Classifier,N_Selected,train_time_s,inference_time_ms,peak_mem_mb
0,Without FS,GaussianNB,24.0,0.569870,150.931727,57.070244
1,Without FS,LogisticRegression,24.0,67.820292,98.193237,57.076390
2,Without FS,DecisionTree,24.0,1.645224,52.736277,57.077119
3,Without FS,MLP,24.0,193.949444,180.701969,57.084051
4,Without FS,RandomForest,24.0,81.879798,8703.595383,57.134888
5,Without FS,HistGradientBoosting,24.0,205.572521,4068.767411,57.136985


,Method,Classifier,accuracy,precision_macro,recall_macro,f1_macro,mcc,n_selected,fs_time_s,train_time_s,inference_time_ms,peak_mem_mb
0,Without FS,DecisionTree,0.8906 +- 0.0011,0.8420 +- 0.0048,0.8984 +- 0.0008,0.8556 +- 0.0032,0.8795 +- 0.0011,24.0000 +- 0.0000,0.895 +- 0.075,1.645 +- 0.120,52.736 +- 8.460,57.077 +- 0.003
1,Without FS,GaussianNB,0.2317 +- 0.0069,0.2905 +- 0.0156,0.3146 +- 0.0059,0.2346 +- 0.0085,0.1751 +- 0.0087,24.0000 +- 0.0000,0.895 +- 0.075,0.570 +- 0.062,150.932 +- 10.250,57.070 +- 0.001
2,Without FS,HistGradientBoosting,0.9082 +- 0.0040,0.8854 +- 0.0042,0.9102 +- 0.0044,0.8852 +- 0.0043,0.8991 +- 0.0044,24.0000 +- 0.0000,0.895 +- 0.075,205.573 +- 48.996,4068.767 +- 2113.183,57.137 +- 0.004
3,Without FS,LogisticRegression,0.6829 +- 0.0008,0.6432 +- 0.0040,0.7204 +- 0.0004,0.6623 +- 0.0031,0.6515 +- 0.0008,24.0000 +- 0.0000,0.895 +- 0.075,67.820 +- 8.056,98.193 +- 12.845,57.076 +- 0.003
4,Without FS,MLP,0.8917 +- 0.0018,0.8846 +- 0.0102,0.8837 +- 0.0136,0.8702 +- 0.0032,0.8809 +- 0.0020,24.0000 +- 0.0000,0.895 +- 0.075,193.949 +- 44.516,180.702 +- 71.969,57.084 +- 0.007
5,Without FS,RandomForest,0.9052 +- 0.0007,0.8554 +- 0.0045,0.9112 +- 0.0006,0.8682 +- 0.0029,0.8957 +- 0.0008,24.0000 +- 0.0000,0.895 +- 0.075,81.880 +- 5.834,8703.595 +- 1414.947,57.135 +- 0.009
